# Horn Antenna Mesh with PalaceToolkit

This notebook uses package helpers for a concise horn mesh workflow:
- `Entity` + `run_meshing_pipeline` for robust geometry labeling
- `generate_3d_mesh` and `refine_near_surfaces` for meshing
- `palacetoolkit.viz` for interactive rendering


In [1]:
import gmsh
from palacetoolkit.mesh import (
    Entity, run_meshing_pipeline, generate_3d_mesh, refine_near_surfaces
)
from palacetoolkit.viz import (
    pv_from_meshio, render_multi_mesh, show_viewer, view_mesh,
    COLOR_DIELECTRIC, COLOR_PORT, COLOR_AIR
)
from IPython.display import HTML

In [2]:
# Geometry parameters (meters)
waveguide_length = 0.30
waveguide_width = 9.373e-2
waveguide_height = 4.166e-2
flare_length = 0.84
flare_width = 30.0 * 0.0254
flare_height = 23.8 * 0.0254
freq_ghz = 1.8
mesh_file = "horn_antenna.msh"

c0 = 3e8
wavelength = c0 / (freq_ghz * 1e9)
outer_radius = max(1.8 * wavelength, 1.2 * flare_length)


In [3]:
gmsh.initialize()
gmsh.option.setNumber("General.Verbosity", 2)
gmsh.model.add("horn_entity_pipeline")
occ = gmsh.model.occ

# Inlet rectangle at z=0
p1 = occ.addPoint(-waveguide_width / 2, -waveguide_height / 2, 0)
p2 = occ.addPoint( waveguide_width / 2, -waveguide_height / 2, 0)
p3 = occ.addPoint( waveguide_width / 2,  waveguide_height / 2, 0)
p4 = occ.addPoint(-waveguide_width / 2,  waveguide_height / 2, 0)
l1 = occ.addLine(p1, p2)
l2 = occ.addLine(p2, p3)
l3 = occ.addLine(p3, p4)
l4 = occ.addLine(p4, p1)
inlet_loop = occ.addCurveLoop([l1, l2, l3, l4])
inlet = occ.addPlaneSurface([inlet_loop])

# Aperture rectangle at z=flare_length
p5 = occ.addPoint(-flare_width / 2, -flare_height / 2, flare_length)
p6 = occ.addPoint( flare_width / 2, -flare_height / 2, flare_length)
p7 = occ.addPoint( flare_width / 2,  flare_height / 2, flare_length)
p8 = occ.addPoint(-flare_width / 2,  flare_height / 2, flare_length)
l5 = occ.addLine(p5, p6)
l6 = occ.addLine(p6, p7)
l7 = occ.addLine(p7, p8)
l8 = occ.addLine(p8, p5)
aperture_loop = occ.addCurveLoop([l5, l6, l7, l8])
aperture = occ.addPlaneSurface([aperture_loop])

# Connect inlet and aperture to form horn flare core volume
l9 = occ.addLine(p1, p5)
l10 = occ.addLine(p2, p6)
l11 = occ.addLine(p3, p7)
l12 = occ.addLine(p4, p8)
s1 = occ.addPlaneSurface([occ.addCurveLoop([l1, l10, -l5, -l9])])
s2 = occ.addPlaneSurface([occ.addCurveLoop([l2, l11, -l6, -l10])])
s3 = occ.addPlaneSurface([occ.addCurveLoop([l3, l12, -l7, -l11])])
s4 = occ.addPlaneSurface([occ.addCurveLoop([l4, l9, -l8, -l12])])
flare_core = occ.addVolume([occ.addSurfaceLoop([inlet, aperture, s1, s2, s3, s4])])

# Backward extrusion of inlet to make waveguide core and waveport face
ext = occ.extrude([(2, inlet)], 0, 0, -waveguide_length)
waveguide_core = [t for d, t in ext if d == 3][0]
waveport = ext[0][1]

outer_air = occ.addSphere(0, 0, flare_length * 0.75, outer_radius)

entities = [
    Entity("waveguide_core", dim=3, mesh_order=1, tags=[waveguide_core]),
    Entity("flare_core", dim=3, mesh_order=1, tags=[flare_core]),
    Entity("outer_air", dim=3, mesh_order=2, tags=[outer_air]),
    Entity("waveport", dim=2, mesh_order=0, tags=[waveport]),
]

pg_map = run_meshing_pipeline(entities)
refine_near_surfaces(entities[-1].dimtags, wavelength, ppw_near=15, ppw_far=4, set_as_background=True)

mesh_sizes = {
    "waveguide_core": wavelength / 12,
    "flare_core": wavelength / 12,
    "waveport": wavelength / 18,
    "outer_air": wavelength / 4,
}
generate_3d_mesh(entities, mesh_sizes, mesh_file, optimize=True)
gmsh.option.setNumber("Mesh.MshFileVersion", 2.2)
gmsh.write(mesh_file)
gmsh.finalize()

pg_map


  Physical group 'waveguide_core' (dim=3): pg=1, tags=[2]
  Physical group 'flare_core' (dim=3): pg=2, tags=[1]
  Physical group 'outer_air' (dim=3): pg=3, tags=[3]
  Physical group 'waveport' (dim=2): pg=4, tags=[11]
  Physical group 'outer_air__waveguide_core' (dim=2): pg=5, tags=[7, 8, 9, 10]
  Physical group 'flare_core__waveguide_core' (dim=2): pg=6, tags=[1]
  Physical group 'flare_core__outer_air' (dim=2): pg=7, tags=[3, 2, 4, 5, 6]
  Physical group 'outer_air__None' (dim=2): pg=8, tags=[12]
  4 conductor boundary curves
  ppw_near=15  ppw_far=4
  SizeMin=0.0111 (15 pts/λ)
  SizeMax=0.0417 (4 pts/λ)
  Transition distance: 0 → 0.0417


Mesh saved to horn_antenna.msh
  Nodes: 45263
  Elements: 262207


{'waveport': 4,
 'outer_air__waveguide_core': 5,
 'flare_core__waveguide_core': 6,
 'flare_core__outer_air': 7,
 'outer_air__None': 8,
 'waveguide_core': 1,
 'flare_core': 2,
 'outer_air': 3}

In [4]:
view_mesh(mesh_file)


Loading mesh file: horn_antenna.msh
Groups to render transparent: ['air_none', 'air_plastic_enclosure']



Mesh loaded successfully with 2 cell blocks
Found 22728 triangles total
Physical group tags in mesh: {4: 'waveport', 5: 'outer_air__waveguide_core', 6: 'flare_core__waveguide_core', 7: 'flare_core__outer_air', 8: 'outer_air__None'}


Widget(value='<iframe src="http://localhost:45963/index.html?ui=P_0x701a167eb050_0&reconnect=auto" class="pyvi…

## Notes

`pg_map` is generated directly by the boolean pipeline and is ready to feed into Palace config-generation helpers in follow-up simulation notebooks.
